*Companion notebook for* **AI APIs as Data Tools**, *from* [Web Data Science](https://cuinfoscience.github.io/Web-Data-Science-Book/) *by Brian C. Keegan (INFO 4617/5617, University of Colorado Boulder).*

*Generated from `ch-13-ai-apis.qmd` — the book chapter is the authoritative version. Code cells are provided unexecuted: run them yourself, and expect to install the chapter's libraries and supply your own API keys where noted. Licensed CC BY-NC-SA 4.0.*

# AI APIs as Data Tools

## Learning Objectives
- Call a model the way you call any other API: one request in, one JSON answer out, checked before you use it
- Turn a PDF, pictures of tables included, into Markdown with a document-understanding API, and wait for a job that answers with a ticket
- Tell what a document parser read from what its model wrote, and check what it read with chapter 9's sums
- Ask a language model for JSON in a shape you define with a JSON Schema, and compare its answers with a ground truth
- Cut documents into chunks, embed them, and find passages by meaning with cosine similarity
- Answer a question from retrieved passages with the IDs of its sources, or with null when the passages don't say
- Read how MCP servers and skills describe tools to a model, and record the costs, limits, and model IDs a run depends on

## Models as Functions, Not Chatbots

@sec-pdfs ended at a picture. The last page of the City of Boulder's December 2024 revenue report, Exhibit 3, is an image of a table, and beside it the city printed a note: "Please contact sales tax staff … for a screen readable version." pdfplumber read every other table in the report exactly, from the text layer, and Tesseract read Exhibit 3 with a few stray marks. Chapter 9's last section, "Document Understanding," described hosted services that read a whole document with a model, for a fee per page, after you upload it. This chapter uses two of them on the same seven year-end reports, 2018 to 2024, and checks them against chapter 9's figures.

You may know language models as chatbots: you type, it answers, you type again. This chapter never chats. It calls a model the way @sec-government called the Census Bureau: one request, built by your code, and one answer, in a shape your code can read. A conversation leaves you with paragraphs that change from one try to the next. A request that returns JSON leaves you with fields you can put in a DataFrame, compare with a known figure, and report as right or wrong. So every model call in this chapter has three parts: a request, a JSON answer, and a check.

The chapter uses three kinds of API, each one step of a single pipeline:

| Kind of API | Service in this chapter | In | Out | The check |
|---|---|---|---|---|
| Document understanding | [Datalab](https://www.datalab.to)'s Convert API | A PDF | Markdown, one page at a time | A table's sums (chapter 9) |
| Structured generation | A free language model, through [OpenRouter](https://openrouter.ai) | An instruction and one page | JSON in a shape you define | Chapter 9's figures from pdfplumber |
| Embeddings | A free embedding model, through OpenRouter | Passages and a question | A list of numbers for each | Whether the passage found prints the answer |

The last two combine in **retrieval-augmented extraction**: find the passages that answer a question, then ask a model for the answer and the IDs of the passages it came from. Each step's output is the next step's input, so a mistake early in the pipeline travels. The chapter finds two that do.

### What You Send, and to Whom

Every request in this chapter sends a document, or part of one, to a company. Datalab receives the whole PDF. OpenRouter receives each page or passage you send and passes it to the company that runs the model, which for this chapter's free models is mostly NVIDIA. Each of them has its own rules for what it keeps:

- **Datalab** keeps a result until "the output bucket's one-day lifecycle rule removes" it, unless you delete it sooner ([Delete Result](https://documentation.datalab.to/api-reference/delete-result), checked 2026-11-07). A team's data-retention setting governs whether Datalab writes "an extra copy for QA and training."
- **OpenRouter** says it does not store your prompts or responses unless you opt in, and its account settings decide whether your requests may go to providers that train on them, with "separate settings for paid and free models" ([Data Collection](https://openrouter.ai/docs/guides/privacy/data-collection) and [Provider Logging](https://openrouter.ai/docs/guides/privacy/provider-logging), checked 2026-11-07).
- **NVIDIA's free endpoints** carry their own notice, on each model's page on OpenRouter: "please do not upload any confidential information or personal data … Your use is logged for security purposes and to improve NVIDIA products and services" (checked 2026-11-07).

So the rule for this chapter, and for the exercises, is to send only public records that name no private individuals. Boulder's revenue reports qualify: they are published on the city's records portal, and they report taxes by category, not by person. One page needs care even so. Page 1 of each report, the cover, gives a city staff member's work email and phone number, so no output in this chapter prints it, and neither should yours. "Free" doesn't change the rule. A free endpoint is paid for in part by what it logs.

## Missing Manual Reference
For what a language model does with the text you send it, from tokens to sampling, see *Missing Manual* [Chapter 36: How Language Models Work](https://cuinfoscience.github.io/INFO-Missing-Manual/chapters/llm-internals.html). For ways to check a model's output before you trust it, see [Chapter 38: Evaluating and Auditing AI Outputs](https://cuinfoscience.github.io/INFO-Missing-Manual/chapters/evaluating-ai.html).

## A Tour: From a Playground to the API

As in @sec-government and @sec-social, start with the pages a person reads. Both services have one.

**Datalab's Forge.** Datalab's [quickstart](https://documentation.datalab.to/docs/welcome/quickstart) points to [Forge](https://www.datalab.to/app/playground), a playground where you upload a document and see the result, with "no API key required" (checked 2026-11-07). Upload a page of your own PDF there before you write any code: if the Markdown that comes back has your table's rows and columns, the API will give you the same, and if it doesn't, no code will fix it. Datalab's [pricing page](https://www.datalab.to/pricing) is the other page to read first, because each of the API's modes has its own rate.

**OpenRouter's model pages.** Every model on OpenRouter has a page, such as [nvidia/nemotron-3.5-super-vl:free](https://openrouter.ai/nvidia/nemotron-3.5-super-vl:free). On 2026-11-07 that page gave the model's name, its ID, its price ("Free"), its context ("1.0M", a million tokens), the date it was released (October 15, 2026), and the notice about logging quoted above. The ID is the one thing your code needs from the page. Its ending, `:free`, names a free *variant* of a model, served without charge and under its own rate limits.

| On the page | What it tells you | Where it goes in your code |
|---|---|---|
| Forge's result for your document | Whether the service can read your tables at all | Nothing yet: it decides whether to write the code |
| Datalab's rate for each mode | What a page costs: `lite`, `fast`, `balanced`, `accurate` | The `mode` field of the request |
| The model's ID on OpenRouter | Which model answers | The `model` field, or a `models` list |
| `:free` and the price | What each request costs | The ID itself: only `:free` IDs are free |
| The context length | How much text one request can carry | How many pages you send at once |
| The notice on a free endpoint | Who logs what you send | What you send at all |
| The supported parameters (in the API's list of models) | Whether the model accepts a JSON Schema, or tools | `response_format`, `tools` |

Nothing on either page is a key. The next section gets them.

## Two Keys

This chapter needs two keys, both free. Get them before you start, and store each as an environment variable by one of the ways in @sec-government ("Setting Environment Variables"), with the course's names:

- **`DATALAB_KEY`.** Sign up at [datalab.to/auth/sign_up](https://www.datalab.to/auth/sign_up), then create a key on the [API Keys](https://www.datalab.to/app/keys) page. New accounts get a free monthly allowance, with no credit card ([quickstart](https://documentation.datalab.to/docs/welcome/quickstart), checked 2026-11-07). Datalab's own documentation and its Python library call the variable `DATALAB_API_KEY`. This book names it `DATALAB_KEY`, as it named `CENSUS_KEY` and `YOUTUBE_KEY`; if you use Datalab's library, set both.
- **`OPENROUTER_KEY`.** Sign up at [openrouter.ai](https://openrouter.ai), then create a key on the [Keys](https://openrouter.ai/settings/keys) page. Requests to `:free` models cost nothing, and this chapter sends no other kind.

Treat each key like a password: never paste it into a notebook cell, a code file, or a message. If one leaks, delete it on its service's keys page and make another. Set the variables, quit Jupyter, open a new terminal, and start Jupyter from it; a Jupyter that was running before you set a variable never receives it (@sec-government, "When a Key Says `MISSING`"). Then check both:

In [ ]:
import os
import re
import io
import json
import time
import requests
import numpy as np
import pandas as pd
import pdfplumber

# Say whether each key is set, without ever printing the key itself
for name in ["DATALAB_KEY", "OPENROUTER_KEY"]:
    print(f"{name}: {'set' if os.environ.get(name) else 'MISSING'}")
datalab_key = os.environ.get("DATALAB_KEY")
openrouter_key = os.environ.get("OPENROUTER_KEY")

# Every request in this chapter says who is asking, as chapter 1's did; put your own address here
HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
# DATALAB_KEY: set
# OPENROUTER_KEY: set

If either says `MISSING` and you can't restart Jupyter now, chapter 11's `getpass()` cell asks for a missing key without saving it in the notebook. Then set the two services' addresses, and the book's copies of the files:

In [ ]:
datalab_base = "https://www.datalab.to/api/v1"
openrouter_base = "https://openrouter.ai/api/v1"
# Chapter 9's copies of the city's PDFs, and this chapter's copies of Datalab's Markdown of them
book = "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-09/"
saved = "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-13/"

# Each service reads its key from its own header; **HEADERS adds the User-Agent to each
datalab_headers = {"X-API-Key": datalab_key, **HEADERS}
openrouter_headers = {"Authorization": f"Bearer {openrouter_key}", **HEADERS}

Unlike the Census Bureau's key in chapter 11, these keys travel in headers, not in the URL, so a printed `response.url` doesn't reveal them. A printed `datalab_headers` does: print responses, never the headers you sent.

## Document Understanding with Datalab

Datalab, which makes [Marker](https://github.com/datalab-to/marker) and [Chandra](https://github.com/datalab-to/chandra), two of the parsers in chapter 9's table of tools, runs its models as a hosted service. Its [Convert API](https://documentation.datalab.to/docs/recipes/conversion/conversion-api-overview) takes a document and returns Markdown, HTML, JSON, or chunks. This section sends it the December 2024 report, reads what comes back, and checks it.

### Submitting a Document

A request to `/convert` is a `POST` with form fields. Two ways to name the document:

- **`file`**: upload the PDF from your computer, as `files=` in `requests`. Use this for a document that isn't on the web.
- **`file_url`**: give the address of a PDF on the web, and Datalab downloads it. The address must be public: Datalab's quickstart lists "using `file_url` with a private/authenticated URL" among the common mistakes. This chapter uses the book's copies on GitHub, so a class doesn't send the city's server a request each.

The other fields say what to make. `output_format` is `markdown`. `mode` chooses how hard Datalab works, and what it costs: `lite` reads a born-digital page from its text layer with fast OCR and makes no passes with a language model, `fast` and `balanced` run its models, and `accurate` runs its strongest ([Convert Document](https://documentation.datalab.to/api-reference/convert-document), checked 2026-11-07). `paginate` marks where each page starts, which the next steps need. Form fields are text, so `"true"` is a string.

In [ ]:
# The December 2024 report, by its address; Datalab downloads it from there
data = {"file_url": book + "revenue-report-2024-12.pdf", "output_format": "markdown",
        "mode": "balanced", "paginate": "true"}
response = requests.post(f"{datalab_base}/convert", data=data, headers=datalab_headers, timeout=60)
response.raise_for_status()
job = response.json()
print(job)
# {'success': True, 'error': None, 'request_id': 'W2Bdj6Blyn7oO0ZDUgBbUQ', 'request_check_url': 'https://www.datalab.to/api/v1/convert/W2Bdj6Blyn7oO0ZDUgBbUQ', 'versions': None, 'checkpoint_reused': None}

The answer isn't the document. Converting takes seconds for a report like this one and can take minutes for a long document, so Datalab answers at once with a **ticket**: a `request_id`, and `request_check_url`, the address where the result will be. Every API in chapters 10 to 12 answered with the data; this one answers with a promise of it.

### Waiting for the Result

Ask the ticket's address whether the job is done. The [result check](https://documentation.datalab.to/api-reference/convert-result-check) answers with a `status`, `processing` until the job is `complete`:

In [ ]:
response = requests.get(job["request_check_url"], headers=datalab_headers, timeout=60)
response.raise_for_status()
check = response.json()
print(check["status"])
# processing

Asking again by hand gets tedious, so let a loop ask, every two seconds. Chapter 12's loop followed a `cursor` until there wasn't one; this one follows the status until the job is complete. Give it a limit, so that a job that never finishes doesn't stop your notebook forever:

In [ ]:
# Ask again every two seconds, at most 60 times: two minutes
for attempt in range(60):
    if check["status"] == "complete":
        break
    time.sleep(2)
    response = requests.get(job["request_check_url"], headers=datalab_headers, timeout=60)
    response.raise_for_status()
    check = response.json()
print(check["page_count"], "pages,", check["total_cost"], "cents,", round(check["runtime"], 1), "seconds")
# 15 pages, 4.5 cents, 10.5 seconds

The finished answer holds the document, in `check["markdown"]`, and an account of the work: `page_count`, the pages it read; `runtime`, the seconds it spent; and the cost, in cents. Datalab's reference marks `total_cost` as deprecated in favor of `cost_breakdown`, which here was `{'final_cost_cents': 4.5}`. When a result is too large to send directly, the answer gives a signed `result_url` to download it from instead; for these reports it was `None`.

Fifteen pages cost 4.5 cents, 0.3 cents a page. Datalab's [pricing page](https://www.datalab.to/pricing) listed, on 2026-11-07, \$4 per 1,000 pages for `fast` and `balanced`, \$0.50 for `lite`, and \$10 for `accurate`, so this bill was three-quarters of the list rate. The same page offers 25% off for opting into data retention: check your account's setting, and what it lets Datalab keep, before you send anything. The free allowance was \$10 a month on a personal email (\$20 on a work email), about 2,500 pages at the list rate for `balanced`, with at most 25 requests a minute. All seven year-end reports, converted the same way on 2026-11-07 for `data/ch-13/`, came to 106 pages and 31.8 cents.

Two things about this run are worth knowing before you repeat it. The first time this chapter sent the request above, the job was complete at the first check, for 0 cents, in 0.1 seconds: the same file had been converted from the same account earlier that day, and Datalab answered from its cache. The output above comes from a second request with `"skip_cache": "true"` added to `data`, which forces a new conversion and bills for it. And a model's output can differ from one conversion to the next, which "What the Model Wrote" shows.

Each conversion takes the same two steps, so gather them into a function for the rest of the chapter:

In [ ]:
def convert(data):
    """Submit a document to Datalab's Convert API, wait for it, and return the finished result.

    Parameters
    ----------
    data : dict
        The request's form fields, such as file_url, output_format, mode, and paginate

    Returns
    -------
    dict
        The result check's answer, with the document in "markdown"
    """
    response = requests.post(f"{datalab_base}/convert", data=data, headers=datalab_headers, timeout=60)
    response.raise_for_status()
    ticket = response.json()["request_check_url"]
    for attempt in range(60):  # at most two minutes
        time.sleep(2)
        response = requests.get(ticket, headers=datalab_headers, timeout=60)
        response.raise_for_status()
        result = response.json()
        if result["status"] == "complete":
            return result
    raise TimeoutError(f"Not complete after two minutes; the result will be at {ticket}")

### Reading the Markdown

With `paginate`, each page of the Markdown starts with its number in braces, counting from 0, and a line of dashes, such as `{14}------------------------------------------------`. [`re.split()`](https://docs.python.org/3/library/re.html#re.split) on that marker gives a list of pages. The text before the first marker is empty, so `[1:]` drops it, and `pages[0]` is page 1:

In [ ]:
markdown = check["markdown"]
pages = re.split(r"\n*\{\d+\}-{10,}\n*", markdown)[1:]
print(len(pages), "pages")
# Page 2, the summary: the rows of its table that hold a total; " ".join(line.split()) squeezes out the padding
for line in pages[1].splitlines():
    if "Total" in line:
        print(" ".join(line.split()))
# 15 pages
# | <b>Total Sales &amp; Use Tax based on current economic activity</b> | <b>\$172,401,668</b> | <b>\$171,006,199</b> | <b>-\$1,395,469</b> | <b>-0.81%</b> | <b>-0.81%</b> |
# | <b>Total Sales and Use Tax</b> | <b>\$175,517,129</b> | <b>\$173,908,431</b> | <b>-\$1,608,698</b> | <b>-0.92%</b> | <b>-0.92%</b> |

These are the rows chapter 9 read with pdfplumber, and the figures agree: \$175,517,129 for 2023 and \$173,908,431 for 2024. A Markdown table writes each row on one line, with `|` between cells. Datalab's Markdown also carries some HTML: `<b>` for bold, `&amp;` for an ampersand, and `\$`, a dollar sign escaped so that Markdown doesn't read it as the start of mathematics. Any code that turns a cell into a number has to remove all three, along with the commas.

Remember the cover: `pages[0]` holds a staff member's email and phone number. The code in this chapter starts at `pages[1]`, and yours should too.

Now page 15, Exhibit 3, the picture of a table:

In [ ]:
# Page 15, Exhibit 3: only the lines of its table, not the note above it
exhibit_lines = [line for line in pages[14].splitlines() if line.startswith("|")]
print(len(exhibit_lines), "lines of table")
for line in [exhibit_lines[0], exhibit_lines[3]]:
    print(" ".join(line.split())[:100])
# 43 lines of table
# | REVENUE<br>CATEGORY | YEAR | JAN | FEB | MAR | APR | MAY | JUN | JUL | AUG | SEP | OCT | NOV | DEC
# | <b>Retail Sales Tax</b> | 2019 | 8,590,452 | 7,489,624 | 9,606,398 | 8,214,047 | 8,832,480 | 10,73

Datalab read the picture as a Markdown table, with the column headings and the categories in place. The second figure in the 2021 row, which Tesseract read as "7,608.759" in chapter 9, is "7,608,759" here. Whether the other 400-odd figures are right is a question for "Checking What It Read," below.

### What the Model Wrote

A document-understanding model does more than read. Where a page has a chart, Datalab's `balanced` mode writes a caption and a description of it, so that a reader of the Markdown, or a screen reader, learns what the picture shows. Chart 8, on page 9, is a pie chart of the city's marijuana revenue, with each slice's figure printed beside it:

In [ ]:
# Page 9: from Chart 8's title to Chart 9's, leaving out the line that links the image
chart_8 = pages[8][pages[8].find("**Chart 8"):pages[8].find("**Chart 9")]
for line in chart_8.splitlines():
    if line.strip() and not line.startswith("!["):
        print(" ".join(line.split()))
# **Chart 8: Distribution of YTD Marijuana Revenues**
# | Category | Amount |
# |---------------------------------------------------------|-------------|
# | State Share-Back | \$529,220 |
# | Medical Marijuana (3.86% Base) | \$52,937 |
# | Recreational Marijuana (3.86% Base) | \$1,262,688 |
# | Recreational Marijuana (3.5% Additional + 5.0% Excise)* | \$1,341,007 |
# A pie chart showing the distribution of marijuana revenue. The largest slice is Recreational Marijuana (3.86% Base) at \$1,262,688, followed by Recreational Marijuana (3.5% Additional + 5.0% Excise)\* at \$1,341,007. State Share-Back is \$529,220, and Medical Marijuana (3.86% Base) is \$52,937.
# Pie chart: Breakdown of marijuana revenue by category and amount.
# \*Dedicated to the General Fund.

The table is a reading: its four figures are printed on the chart, and pdfplumber finds each of them in the page's text layer. The paragraph under it is writing. It calls \$1,262,688 the largest slice, then says the next is \$1,341,007, which is larger. Every figure in the sentence is right, and the sentence is wrong.

Page 10 goes further. Its bar chart of sugar-sweetened beverage tax by month has no figures on it, only an axis. Datalab wrote a description and a table of the values it estimated from the bars:

In [ ]:
# Page 10: the description, the table's heading, and its December row
for line in pages[9].splitlines():
    if line.startswith(("A bar chart", "| Month", "| December")):
        print(" ".join(line.split()))
# A bar chart showing monthly values from January to December for the years 2019 through 2024. The y-axis represents values in dollars, ranging from \$- to \$600,000. The x-axis shows the months. 2019 is the highest in November, while 2024 is the highest in March and November.
# | Month | 2019 | 2020 | 2021 | 2022 | 2023 | 2024 |
# | December | ~\$370,000 | ~\$225,000 | ~\$335,000 | ~\$355,000 | ~\$325,000 | ~\$285,000 |

The table has 72 estimates, each marked with a `~`, and none of them is printed in the report. The description contradicts the table beside it, too: the table puts November 2024 at about \$250,000, the lowest November of the six years, while the description calls November one of 2024's highest months. The PDF's own text layer shows what the page prints. Read it with pdfplumber, as in chapter 9, from the book's copy:

In [ ]:
response = requests.get(book + "revenue-report-2024-12.pdf", headers=HEADERS, timeout=60)
response.raise_for_status()
# The text layer of every page, from the PDF's bytes in memory
text_layer = []
with pdfplumber.open(io.BytesIO(response.content)) as pdf:
    for page in pdf.pages:
        text_layer.append(page.extract_text() or "")
# Page 10's lines with a dollar sign: everything it prints in dollars
print([line for line in text_layer[9].splitlines() if "$" in line])
# ['$600,000', '$500,000', '$400,000', '$300,000', '$200,000', '$100,000', '$-']

The page prints the axis and nothing else. Keep `text_layer`: "Retrieval-Augmented Extraction" uses it again.

**Two runs, two documents.** This chapter's conversion and the one that made `data/ch-13/`, about twenty minutes apart on 2026-11-07, differ on six of the fifteen pages. On four of them the model wrote differently: Chart 1's table headed its first column "Department" instead of "Category," estimates moved on pages 5 and 10 (on page 10, one went from about \$415,000 to about \$410,000), and descriptions changed on pages 4, 5, and 8. On the other two only the formatting changed, such as bold tags in a header. Every figure Datalab read from the text layer or from the picture of Exhibit 3 was the same both times. What a model reads is stable; what it writes is not.

**`lite` mode.** The cheaper mode makes no passes with a language model, so it writes nothing:

In [ ]:
lite = convert({**data, "mode": "lite"})  # the same fields, with the mode changed
print(lite["page_count"], "pages,", lite["total_cost"], "cents,", round(lite["runtime"], 1), "seconds")
lite_pages = re.split(r"\n*\{\d+\}-{10,}\n*", lite["markdown"])[1:]
# Chart 9, the line chart under Chart 8, as lite mode read it
chart_9 = lite_pages[8][lite_pages[8].find("Chart 9"):]
for line in chart_9.splitlines()[:7]:
    if line.strip():
        print(" ".join(line.split()))
# 15 pages, 0.5625 cents, 1.9 seconds
# Chart 9: Historical Trends for YTD Marijuana Revenues3
# | | | | | |
# |-------------|-------------|-------------------|---------------|-------------|
# | \$6,000,000 | \$5,638,279 | | | |
# | \$5,000,000 | \$5,429,87 | | | |
# | \$4,000,000 | | \$4,372,65 8,918 | | |

(This chapter's run added `"skip_cache": "true"` here too.) `lite` cost an eighth as much and took a fifth of the time. Its Markdown has no descriptions and not one `~`, where `balanced` has 124. It read Exhibit 3 too, and all 30 of its rows pass the check below. But it read Chart 9, whose figures are printed around the lines, as a grid: the axis labels in one column, "\$5,429,87" with its last digit lost, and "\$4,372,65 8,918," two figures run together. `balanced` read the same chart's figures into a clean table. Choose the mode for the pages you need: `lite` for text and plain tables, `balanced` for charts whose figures are printed on them, and in either case, check.

Datalab's reference lists two fields for the writing: `disable_image_captions`, which turns off "synthetic image captions/descriptions," and `fence_synthetic_captions`, which wraps them in HTML comment markers "for easy identification/removal." This chapter didn't test whether either one covers the tables of estimates. If you try them, check the beverage page.

### Checking What It Read

Chapter 9's first check on Exhibit 3 was a sum: each row's twelve months should add up to its total, give or take the rounding of each figure to the dollar. The same check works on Datalab's table. In each line of the table, the cells sit between `|` characters, so `line.split("|")` gives `['', label, year, twelve months, total, ...]`. Two small functions do the work, because the 2018 report, and the exercises, need them again:

In [ ]:
def table_cells(line):
    """Split one line of a Markdown table into its cells, without bold tags, commas, or spaces."""
    cells = []
    for cell in line.split("|"):
        cells.append(cell.replace("<b>", "").replace("</b>", "").replace(",", "").strip())
    return cells

def figure_rows(page):
    """Return the rows of an Exhibit 3 table, as lists of cells: the lines whose third cell is a year."""
    rows = []
    for line in page.splitlines():
        cells = table_cells(line)
        if len(cells) > 16 and re.fullmatch(r"20\d\d", cells[2]):
            rows.append(cells)
    return rows

exhibit = figure_rows(pages[14])
print(len(exhibit), "rows of figures")
# Each category's name starts its first row
labels = []
for cells in exhibit:
    if cells[1]:
        labels.append(cells[1])
print(labels)
# 30 rows of figures
# ['Retail Sales Tax', 'Business Use Tax', 'Construction Use Tax', 'Motor Vehicle Use Tax', 'TOTAL']

Thirty rows: four taxes and their total, each for the six years from 2019 to 2024, with the labels in the order the picture prints them. Now the sums:

In [ ]:
def largest_gap(rows):
    """Return the largest difference, in dollars, between a row's twelve months and its printed total."""
    largest = 0
    for cells in rows:
        months = []
        for cell in cells[3:15]:
            months.append(int(cell))
        gap = abs(sum(months) - int(cells[15]))
        if gap > largest:
            largest = gap
    return largest

print("largest gap between the months and the total:", largest_gap(exhibit), "dollars")
# largest gap between the months and the total: 2 dollars

Every row adds up within \$2, as Tesseract's rows did in chapter 9. A digit misread anywhere but the last place would miss by \$10 or more. Datalab's table also passes a second check from chapter 9, two printings of the same figure: Exhibit 3's Retail Sales Tax total for 2024, 142,513,015, is the Sales Tax figure in page 13's Exhibit 1, which comes from the text layer.

The 2018 report is older, and its Exhibit 3 is a table of text, not a picture. Its Markdown is in the book's saved copy, so checking it costs nothing:

In [ ]:
response = requests.get(saved + "revenue-report-2018.md", headers=HEADERS, timeout=60)
response.raise_for_status()
pages_2018 = re.split(r"\n*\{\d+\}-{10,}\n*", response.text)[1:]
rows_2018 = figure_rows(pages_2018[14])
print(len(pages_2018), "pages;", len(rows_2018), "rows of figures; largest gap:", largest_gap(rows_2018), "dollars")
# 15 pages; 28 rows of figures; largest gap: 0 dollars

Four categories for the seven years from 2012 to 2018, and every row exact to the dollar. Put the tools side by side on the 2024 picture of a table, with the same check:

| Tool | Rows of 30 read whole | Every whole row adds up | Cost |
|---|---|---|---|
| Tesseract, with chapter 9's pattern | 28; 2 set aside for stray marks | Yes, within \$2 | Free, on your computer |
| Docling 2.139.0 (chapter 9) | 21; rows merged or emptied | Yes, but two with the wrong label | Free, about a minute on a laptop |
| Datalab, `lite` | 30 | Yes, within \$2 | 0.5625 cents for the report |
| Datalab, `balanced` | 30 | Yes, within \$2 | 4.5 cents for the report |

Sums catch a misread digit. They can't catch a right row with the wrong label, which is how Docling failed in chapter 9, or a description that contradicts its figures. Datalab's labels are in the right order here; you know that from reading them, not from the sums.

## A Model as a Function: Structured Output

Datalab turned the reports into text. Now a language model reads that text and returns the figures you ask for, as JSON. This section asks for four fields from each report's summary page: the month and year the report covers, and the totals row's two figures, the year before and the report's own year. Chapter 9 read those same figures with pdfplumber, and checked every one, so its numbers are the **ground truth**: the answer you compare the model's with.

The rest of the chapter reads the book's saved copies of the seven reports' Markdown, made with `convert()` as above on 2026-11-07, so it needs no Datalab key and costs nothing to re-run.

### One Request, One JSON Object

OpenRouter sends one kind of request, OpenAI's chat-completions format, to models from many companies ([quickstart](https://openrouter.ai/docs/quickstart)). Which models are free, and what each accepts, is in its list of models, which needs no key:

In [ ]:
response = requests.get(f"{openrouter_base}/models", headers=HEADERS, timeout=60)
response.raise_for_status()
models = response.json()["data"]
free = [model for model in models if model["id"].endswith(":free")]
print(len(models), "models;", len(free), "free")
# The free models whose list of supported parameters includes structured outputs
print([model["id"] for model in free if "structured_outputs" in model["supported_parameters"]])
# 460 models; 12 free
# ['liquid/lfm-3-1.7b:free', 'recursive/flint-v1.0:free', 'xingchen-agi/xing4.0-29b-a4b:free', 'nvidia/nemotron-3.5-super-vl:free', 'dots-studio/dots-3-note-preview:free']

The list changes from week to week; this is 2026-11-07's. This chapter uses three free models, checked that day, in order of preference: NVIDIA's Nemotron 3.5 Super VL, Google's Gemma 4 31B, and Dots Studio's Dots3-Note Preview. Gemma's entry lists `response_format` but not `structured_outputs`, so OpenRouter doesn't list it among the models that enforce a schema; the first and third are on that list.

Read the summary page of the December 2024 report, the first page whose text holds the table's last row:

In [ ]:
response = requests.get(saved + "revenue-report-2024-12.md", headers=HEADERS, timeout=60)
response.raise_for_status()
pages = re.split(r"\n*\{\d+\}-{10,}\n*", response.text)[1:]
summary = [page for page in pages if "Total Sales and Use Tax" in page][0]
print("page", pages.index(summary) + 1, "|", len(summary), "characters")
# page 2 | 4902 characters

Now the shape of the answer. A **JSON Schema** describes JSON the way a form describes what goes in each box: an object with these four properties, each of this type, all of them required, and no others. `month` must be one of twelve strings (`enum`), the years and totals whole numbers (`integer`). Each property's `description` says what belongs in it, in words:

In [ ]:
MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
SCHEMA = {
    "type": "object",
    "properties": {
        "month": {"type": "string", "enum": MONTHS,
                  "description": "The month the year-to-date figures run through"},
        "year": {"type": "integer",
                 "description": "The report's current year: the later of the two years the table compares"},
        "total_prior": {"type": "integer",
                        "description": "The table's last row, Total Sales and Use Tax, for the earlier year, in dollars"},
        "total_current": {"type": "integer",
                          "description": "The table's last row, Total Sales and Use Tax, for the report's current year, in dollars"},
    },
    "required": ["month", "year", "total_prior", "total_current"],
    "additionalProperties": False,
}
INSTRUCTION = ("You extract figures from one page of a City of Boulder revenue report. "
               "Copy numbers exactly as printed, without $ or commas. Use only the page.")

The request's body has four parts, and the key goes in `openrouter_headers`:

- **`messages`**: a list with two items. The `system` message is the instruction; the `user` message is the page. That is the whole conversation, and no request in this chapter adds a third.
- **`response_format`**: `type` `json_schema`, with the schema, a `name` for it, and `strict` set to `True`, which asks the model to follow the schema exactly ([Structured Outputs](https://openrouter.ai/docs/guides/features/structured-outputs)). OpenRouter's page adds that enforcement "varies by provider," so "exact compliance is not guaranteed on every endpoint."
- **`temperature`**: `0` asks the model for its most likely answer at every step, rather than a sample. "Non-determinism," below, shows that this isn't the same as getting the same answer every time.
- **`models`**: a list, in place of a single `model`. If the first returns an error, including the 429 of a rate limit, OpenRouter tries the next ([Model Fallbacks](https://openrouter.ai/docs/guides/routing/model-fallbacks)). The page warns that a paid model in the list is billed at its price: "To keep a request free, list only :free models."

In [ ]:
MODELS = ["nvidia/nemotron-3.5-super-vl:free", "google/gemma-4-31b-it:free", "dots-studio/dots-3-note-preview:free"]
body = {"models": MODELS, "temperature": 0,
        "messages": [{"role": "system", "content": INSTRUCTION}, {"role": "user", "content": summary}],
        "response_format": {"type": "json_schema",
                            "json_schema": {"name": "totals", "strict": True, "schema": SCHEMA}}}
response = requests.post(f"{openrouter_base}/chat/completions", json=body, headers=openrouter_headers, timeout=180)
response.raise_for_status()
result = response.json()
print(result["model"], "|", result["provider"])
# The answer is a string of JSON; json.loads() turns it into a dictionary
answer = json.loads(result["choices"][0]["message"]["content"])
print(answer)
usage = result["usage"]
print(usage["prompt_tokens"], "tokens in,", usage["completion_tokens"], "out, of them",
      usage["completion_tokens_details"]["reasoning_tokens"], "reasoning; cost:", usage["cost"])
# nvidia/nemotron-3.5-super-vl:free | Nvidia
# {'month': 'December', 'year': 2024, 'total_prior': 175517129, 'total_current': 173908431}
# 1568 tokens in, 2451 out, of them 2398 reasoning; cost: 0

The response says which model answered, `model`, and which company ran it, `provider`: with a fallback list, those can change from one request to the next, so record them with every answer. The answer itself is in `choices[0]["message"]["content"]`, as a string, and `json.loads()` makes it a dictionary.

`usage` counts **tokens**, the pieces of words a model reads and writes. The instruction and the page came to 1,568 tokens. The answer was 2,451, but the JSON holds about fifty; the other 2,398 were *reasoning tokens*, text the model wrote to itself before answering. They are in the response too, in `choices[0]["message"]["reasoning"]`, and on a paid model you pay for them as for any other output. Here they cost nothing, because the model is free.

### Checking Against Ground Truth

Chapter 9 read the same row with pdfplumber, by position, and checked each figure against the report's own change column and against the next year's printing ("The Time Series" and "Two Printings of the Same Year"). Those figures, for each report's year and the year before, are the ground truth:

In [ ]:
# Chapter 9's figures for the last row, Total Sales and Use Tax: the year before, and the report's own year
TRUTH = {
    "revenue-report-2018": (134538892, 140648710), "revenue-report-2019": (140609751, 141954650),
    "revenue-report-2020": (141954650, 135195413), "revenue-report-2021-12": (135195413, 152302653),
    "revenue-report-2022-12": (152302653, 169112615), "revenue-report-2023-12": (169112615, 175517129),
    "revenue-report-2024-12": (175517129, 173908431),
}
prior, current = TRUTH["revenue-report-2024-12"]
print("prior:  ", answer["total_prior"], prior, answer["total_prior"] == prior)
print("current:", answer["total_current"], current, answer["total_current"] == current)
print(answer["year"], answer["month"])
# prior:   175517129 175517129 True
# current: 173908431 173908431 True
# 2024 December

All four fields are right. Note one thing about `TRUTH`: the 2018 report gives 2018's total as \$140,648,710, and the 2019 report gives it as \$140,609,751. `TRUTH` keeps each report's own printing, because the model reads one report at a time.

### Seven Reports

One right answer is a start. Send the same request for each of the seven reports, and count. The request is the same each time except for the page, so it becomes a function:

In [ ]:
def extract_totals(page):
    """Send one page to a model with SCHEMA and INSTRUCTION; return the answer, as a dictionary, and the model that gave it."""
    body = {"models": MODELS, "temperature": 0,
            "messages": [{"role": "system", "content": INSTRUCTION}, {"role": "user", "content": page}],
            "response_format": {"type": "json_schema",
                                "json_schema": {"name": "totals", "strict": True, "schema": SCHEMA}}}
    response = requests.post(f"{openrouter_base}/chat/completions", json=body, headers=openrouter_headers, timeout=180)
    response.raise_for_status()
    result = response.json()
    return json.loads(result["choices"][0]["message"]["content"]), result["model"]

rows = []
for report, (prior, current) in TRUTH.items():
    response = requests.get(saved + report + ".md", headers=HEADERS, timeout=60)
    response.raise_for_status()
    pages = re.split(r"\n*\{\d+\}-{10,}\n*", response.text)[1:]
    summary = [page for page in pages if "Total Sales and Use Tax" in page][0]
    answer, model = extract_totals(summary)
    rows.append({"report": report[15:], "model": model, **answer,
                 "prior_ok": answer["total_prior"] == prior, "current_ok": answer["total_current"] == current})
    time.sleep(2)  # a pause between requests; free models allow 20 a minute
results = pd.DataFrame(rows)
print(results.drop(columns="model").to_string(index=False))
#  report    month  year  total_prior  total_current  prior_ok  current_ok
#    2018 December  2018    140648710      139331838     False       False
#    2019  January  2020    141954650      139548151     False       False
#    2020 December  2020    141954650      135195413      True        True
# 2021-12 December  2021    135195413      152302653      True        True
# 2022-12 December  2022    152302653      169112615      True        True
# 2023-12 December  2023    169112615      175517129      True        True
# 2024-12 December  2025    175517129      173908431      True        True

Every answer came from the first model in the list, and every answer is valid JSON in the schema's shape. Five of the seven reports' totals match chapter 9. Two don't, and the year is wrong in two rows. Print the 2018 summary page and find where the wrong figures came from. The whole table is on the page, totals row included, and both wrong figures are in it, in the 2018 column: \$140,648,710 is 2018's own total, returned as the year before, and \$139,331,838 is the row above the totals, "Total Sales & Use Tax based on current economic activity," a subtotal that leaves out audit revenue, returned as 2018's total. The model moved one column over and one row up. The 2019 answer made the same two moves. Its year and month, 2020 and January, come from a sentence on the page: the revenue "was received by the city through January 2020." The 2024 answer's year, 2025, comes from the same sentence in that report. Chapter 9's code read these rows by their labels and positions; the model chose by what the words seemed to mean.

The responses hold a clue to why. Each one carries the model's reasoning, 4,000 to 12,000 characters of it for these reports. Not one of the eight responses so far, this section's seven and the single request above, mentions `total_prior`, `total_current`, or JSON. In the 2018 reasoning, the model planned to answer with a list of every figure on the page, and wrote out the summary table along the way with the totals row right. The JSON came afterward: the provider held the model's output to the schema's shape as it wrote it, and the descriptions that say which row and which year each field means seem never to have reached the model. The schema fixed the answer's form, and nothing told the model what the fields meant.

So tell it. Add the schema to the instruction, as text, and run the seven reports again:

In [ ]:
# The same instruction, with the schema itself added as text; extract_totals() reads INSTRUCTION each time it runs
INSTRUCTION = INSTRUCTION + " Answer with JSON that fits this schema: " + json.dumps(SCHEMA)

Run the loop above again, unchanged:

In [ ]:
# Run the loop that starts "rows = []" again; it printed:
#  report    month  year  total_prior  total_current  prior_ok  current_ok
#    2018 December  2018    134538892      140648710      True        True
#    2019 December  2019    140609751      141954650      True        True
#    2020 December  2020    141954650      135195413      True        True
# 2021-12 December  2021    135195413      152302653      True        True
# 2022-12 December  2022    152302653      169112615      True        True
# 2023-12 December  2023    169112615      175517129      True        True
# 2024-12 December  2024    175517129      173908431      True        True

Seven of seven totals, years, and months. The reasoning shrank too: 450 to 720 tokens per report, where the first run spent 2,400 to 6,000, and the loop took under a minute instead of nearly three. A second try of the 2018 page five minutes later repeated both results: wrong with the plain instruction (this time with January 2019 as the month and year), and right with the schema in it. That is two runs of one model on seven pages, enough to change your practice and not enough to promise it works for every model. Describe the fields in the instruction as well as in the schema, and keep the check that found the problem.

### Non-determinism

The single request at the start of this section returned 2024 as the year. The loop, three minutes later, returned 2025 for the same page with the same request. Send it three more times, with the instruction as it was:

In [ ]:
INSTRUCTION = ("You extract figures from one page of a City of Boulder revenue report. "
               "Copy numbers exactly as printed, without $ or commas. Use only the page.")
for attempt in range(3):
    answer, model = extract_totals(summary)  # summary: the last page the loop found, 2024's
    print(model, answer)
    time.sleep(2)
# nvidia/nemotron-3.5-super-vl:free {'month': 'December', 'year': 2025, 'total_prior': 175517129, 'total_current': 173908431}
# nvidia/nemotron-3.5-super-vl:free {'month': 'December', 'year': 2025, 'total_prior': 175517129, 'total_current': 173908431}
# nvidia/nemotron-3.5-super-vl:free {'month': 'December', 'year': 2025, 'total_prior': 175517129, 'total_current': 173908431}

Five identical requests, at temperature 0, within twelve minutes: one said 2024 and four said 2025. The three above agree with each other, but their reasoning ran to 2,396, 2,403, and 3,314 tokens: the model took a different path each time and arrived at the same wrong year. Temperature 0 picks the most likely next token, but the provider's hardware computes "most likely" with arithmetic whose rounding can depend on what else it is serving at the moment, so a close call can go either way. A fallback list adds a second source of change: a busy first model hands your request to a different model altogether. A study of 829 code-generation problems found the same prompt returning different code from one request to the next, even at temperature 0 [@ouyang2024empirical], and a commercial model's answers to the same questions changed between versions released three months apart [@chen2024how].

So a model's answer is an observation, not a fact. Record what you need to say what you observed: the model ID that answered (`result["model"]`), the provider, the date and time, the request body, and the raw response. Then, where you can, run the request more than once and report how often the answers agreed.

## Embeddings and Retrieval

Structured output answered a question you could point at: this page, this row. Often you can't point. Which of the seven reports' 106 pages explains the marijuana share-back, or the fall in 2020? Search by keyword finds pages with your words; search by meaning finds pages about your question, in other words. An **embedding** makes that possible: a model turns a passage into a long list of numbers, a *vector*, so that passages about the same thing get similar lists. A question gets a vector too, and the passages whose vectors are closest to it are the ones to read.

### Chunks

An embedding model reads a limited amount of text at once. Send it a whole page of Exhibit 3, from the saved 2024 Markdown that the loop above left in `pages`, and it refuses:

In [ ]:
EMBEDDER = "nvidia/nemotron-3-embed-1b:free"
response = requests.post(f"{openrouter_base}/embeddings", json={"model": EMBEDDER, "input": [pages[14]]},
                         headers=openrouter_headers, timeout=60)
print(len(pages[14]), "characters:", response.status_code, response.json()["error"]["message"])
# 16333 characters: 422 HTTP 422: {"object":"error","message":"input length 7654 exceeds model maximum 4096; set truncate=END or START to truncate long inputs","type":"invalid_request_error"}

The page is 7,654 tokens; the model takes at most 4,096. Its entry in OpenRouter's list of embedding models gave a context of 32,768 on 2026-11-07, so the error, not the list, is the limit to trust. Truncating, as the message suggests, would throw away the rest of the page. Instead, cut each page into **chunks** of at most about 2,500 characters, about 600 tokens, breaking only between lines, so that a row of a table stays whole. Give each chunk an ID that says where it came from, the report, the page, and the chunk, such as `2024-12 p.9 #1`:

In [ ]:
REPORTS = ["revenue-report-2018", "revenue-report-2019", "revenue-report-2020", "revenue-report-2021-12",
           "revenue-report-2022-12", "revenue-report-2023-12", "revenue-report-2024-12"]
chunks = []
for report in REPORTS:
    response = requests.get(saved + report + ".md", headers=HEADERS, timeout=60)
    response.raise_for_status()
    pages = re.split(r"\n*\{\d+\}-{10,}\n*", response.text)[1:]
    for number, page in enumerate(pages, start=1):
        text = ""
        part = 1
        for line in page.splitlines(keepends=True):
            # Start a new chunk when this line would take the current one past 2,500 characters
            if len(text) + len(line) > 2500 and text:
                chunks.append({"id": f"{report[15:]} p.{number} #{part}", "report": report, "text": text})
                text = ""
                part = part + 1
            text += line
        chunks.append({"id": f"{report[15:]} p.{number} #{part}", "report": report, "text": text})
chunks = pd.DataFrame(chunks)
print(len(chunks), "chunks from", chunks["report"].nunique(), "reports")
# 243 chunks from 7 reports

Most pages are one chunk; Exhibit 3 and the long tables of Exhibit 1 are several. A chunk that starts inside a table loses the table's header, so a chunk such as `2024-12 p.15 #3` is rows of figures with no column names. That is the price of the limit, and a reason to keep chunk IDs: they lead you back to the whole page.

### Embedding and Cosine Similarity

[OpenRouter's embeddings endpoint](https://openrouter.ai/docs/api/reference/embeddings) takes a `model` and an `input`, a list of texts, and returns one vector for each. Embed one chunk first, the 2018 summary:

In [ ]:
response = requests.post(f"{openrouter_base}/embeddings", json={"model": EMBEDDER, "input": [chunks["text"][1]]},
                         headers=openrouter_headers, timeout=60)
response.raise_for_status()
vector = response.json()["data"][0]["embedding"]
print(len(vector), "numbers:", [round(x, 4) for x in vector[:4]], "...")
print(response.json()["usage"]["prompt_tokens"], "tokens")
# 2048 numbers: [0.0494, 0.041, -0.0287, -0.0178] ...
# 588 tokens

The vector has 2,048 numbers, and none of them means anything alone. What carries meaning is how two vectors compare. **Cosine similarity** measures the angle between two vectors: 1 when they point the same way, near 0 when they have nothing in common. For vectors scaled to length 1, it is their *dot product*, the sum of their numbers multiplied pair by pair, and NumPy's `@` computes it. So scale every vector to length 1 once, and a single `vectors @ query` scores every chunk against a question at once.

Embed all 243 chunks, 20 to a request, with a pause between requests; on 2026-11-07 this model took batches of 20:

In [ ]:
vectors = []
for start in range(0, len(chunks), 20):
    batch = chunks["text"][start:start + 20].tolist()
    response = requests.post(f"{openrouter_base}/embeddings", json={"model": EMBEDDER, "input": batch},
                             headers=openrouter_headers, timeout=120)
    response.raise_for_status()
    for item in response.json()["data"]:
        vectors.append(item["embedding"])
    time.sleep(1)
vectors = np.array(vectors)
# Scale each row to length 1: divide it by its length, np.linalg.norm
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
print(vectors.shape)
# (243, 2048)

One rule follows from how embeddings work: **never mix embedding models.** Each model places passages in its own space, and a vector from one model compared with a vector from another is meaningless, even when both have 2,048 numbers. So `EMBEDDER` gets no fallback list. If the model you used is retired, embed every chunk again with a new one, and record which model made the vectors you saved.

### What Embeddings Find

A question becomes a vector the same way, and the chunks with the highest scores are the best matches:

In [ ]:
def embed_question(question):
    """Return a question's embedding, from EMBEDDER, scaled to length 1."""
    response = requests.post(f"{openrouter_base}/embeddings", json={"model": EMBEDDER, "input": [question]},
                             headers=openrouter_headers, timeout=60)
    response.raise_for_status()
    query = np.array(response.json()["data"][0]["embedding"])
    return query / np.linalg.norm(query)

chunks["score"] = vectors @ embed_question("How much revenue did the state share back from marijuana taxes?")
print(chunks.nlargest(5, "score")[["id", "score"]].round(3).to_string(index=False))
#             id  score
#    2019 p.8 #1  0.467
#    2020 p.9 #1  0.466
#    2018 p.9 #1  0.464
# 2023-12 p.9 #1  0.461
# 2022-12 p.9 #1  0.459

The five best chunks are the marijuana page of five different reports, and each of them answers the question for its own year. The search found the topic. It didn't find 2024's answer, because the question didn't name a year, and every year's page is about the same thing: the 2024 page ranked seventh. Ask about a figure, and the search finds something else:

In [ ]:
chunks["score"] = vectors @ embed_question("Which revenue came to $529,220 in 2024?")
print(chunks.nlargest(5, "score")[["id", "score"]].round(3).to_string(index=False))
#              id  score
# 2024-12 p.15 #6  0.339
# 2024-12 p.15 #2  0.331
# 2024-12 p.15 #7  0.329
#  2024-12 p.9 #1  0.309
# 2024-12 p.15 #3  0.297

The figure appears in exactly one chunk, `2024-12 p.9 #1`, which ranked fourth. The three above it are chunks of Exhibit 3: rows of numbers, close in meaning to a question about a number, with no \$529,220 in them. An embedding captures what a passage is about. One figure among a hundred barely moves it, which is why chapter 9's search by pattern, `re.findall()` or `"529,220" in text`, finds a figure and an embedding doesn't.

Two fixes follow. Name in the question what the passage would say, such as the year. And filter before you score: if you know the answer is in the 2024 report, keep only its chunks, with `chunks[chunks["report"] == "revenue-report-2024-12"]`, and rank those. A filter you can state is a filter you can check, which a similarity score is not.

## Retrieval-Augmented Extraction

**Retrieval-augmented generation**, or RAG, combines the two halves of this chapter: retrieve the passages that best match a question, then ask a model to answer from them [@lewis2020retrieval]. This chapter keeps the generation to a figure and its citations. The model must return JSON with two fields: `value`, the figure, or **null** when no passage prints it; and `source_ids`, the IDs of the passages it came from. Null is the model's way to say "the passages don't say," and a pipeline that can't say that will fill the gap with something.

In [ ]:
ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "value": {"type": ["integer", "null"],
                  "description": "The figure, in dollars, exactly as printed in a passage; null if no passage prints it"},
        "source_ids": {"type": "array", "items": {"type": "string"},
                       "description": "The IDs of the passages that print the figure; empty if value is null"},
    },
    "required": ["value", "source_ids"],
    "additionalProperties": False,
}
# The instruction, with the schema in it as text, as "Seven Reports" found works better
ANSWER_INSTRUCTION = ("Answer the question using only the passages, each headed by its ID. "
                      "Give a figure only if a passage prints it; an estimate read from a chart doesn't count. "
                      "If no passage prints it, answer null. Answer with JSON that fits this schema: "
                      + json.dumps(ANSWER_SCHEMA))

def answer_with_sources(question, report):
    """Score the chunks against a question, send one report's best three to a model, and return its answer."""
    chunks["score"] = vectors @ embed_question(question)
    top = chunks[chunks["report"] == report].nlargest(3, "score")
    print(top[["id", "score"]].round(3).to_string(index=False))
    # Each passage headed by its ID, with a blank line between passages
    passages = "\n\n".join(f"{row.id}: {row.text}" for row in top.itertuples())
    body = {"models": MODELS, "temperature": 0,
            "messages": [{"role": "system", "content": ANSWER_INSTRUCTION},
                         {"role": "user", "content": f"Question: {question}\n\n{passages}"}],
            "response_format": {"type": "json_schema",
                                "json_schema": {"name": "answer", "strict": True, "schema": ANSWER_SCHEMA}}}
    response = requests.post(f"{openrouter_base}/chat/completions", json=body, headers=openrouter_headers, timeout=180)
    response.raise_for_status()
    result = response.json()
    print(result["model"])
    return json.loads(result["choices"][0]["message"]["content"])

print(answer_with_sources("How much revenue did the state share back from marijuana taxes?", "revenue-report-2024-12"))
#              id  score
#  2024-12 p.9 #1  0.452
#  2024-12 p.9 #2  0.364
# 2024-12 p.11 #2  0.357
# nvidia/nemotron-3.5-super-vl:free
# {'value': 529220, 'source_ids': ['2024-12 p.9 #1']}

Filtered to the 2024 report, the marijuana page ranks first, and the model returns \$529,220 with that page as its source. The check that a figure needs is whether it is printed where the source says. `text_layer`, the PDF's own text from "What the Model Wrote," answers that without any model:

In [ ]:
def printed_on(value):
    """Return the pages of the PDF's text layer that print a figure, written with commas."""
    found = []
    for number, text in enumerate(text_layer, start=1):
        if f"{value:,}" in text:
            found.append(number)
    return found

print("529,220 is printed on pages", printed_on(529220))
# 529,220 is printed on pages [9]

Page 9, the page `2024-12 p.9 #1` came from. Now a question whose answer the report doesn't print: the sugar-sweetened beverage tax for December 2024, which page 10 shows only as a bar.

In [ ]:
print(answer_with_sources("How much sugar-sweetened beverage tax did the city collect in December 2024?",
                          "revenue-report-2024-12"))
print("285,000 is printed on pages", printed_on(285000))
#              id  score
#  2024-12 p.2 #1  0.469
# 2024-12 p.10 #1  0.467
#  2024-12 p.1 #1  0.456
# nvidia/nemotron-3.5-super-vl:free
# {'value': 285000, 'source_ids': ['2024-12 p.10 #1']}
# 285,000 is printed on pages []

The right answer is null. The model returned \$285,000, with a source, and the source does print it: `~\$285,000` is the December 2024 cell of the table Datalab wrote beside the bar chart. The model's reasoning, returned with the answer, weighed the instruction and decided: "The table is not a chart. It's a table. The values are printed. So it counts." Given only the passages, that is a fair reading; nothing in the Markdown says that a model wrote the table, not the city. The text-layer check catches it, because the PDF prints no 285,000 anywhere.

An earlier run on the same day, with an instruction that didn't include the schema, did answer null, with the same source. Its reasoning had settled on "~\$285,000," tilde and all, and the schema's integer-or-null had no way to write a tilde. The null was right by accident. Two things follow. The error belongs to the pipeline's first step: Datalab wrote an estimate in a form the third step can't tell from a reading, and no instruction to the third step fixes that. And `source_ids` don't make an answer true. They tell you where to look, and the looking is still yours: the text layer for a born-digital PDF, the image for a scan, a person for anything else.

Look again at the passages for the beverage question: the third, `2024-12 p.1 #1`, is the cover, with the staff member's contact details, sent to the model as a passage. Nothing printed it, but it went. If you build on this pipeline, drop each report's first page from `chunks` before you embed, with `chunks[~chunks["id"].str.contains(r" p\.1 #")]`.

## Tools, MCP, and Skills

Everything so far has your code in charge: it chooses the page, sends it, and checks what comes back. Some programs hand part of that control to the model. **Tool calling** lets a request list `tools`, each with a name, a description, and its parameters described by a JSON Schema; the model may answer with a *tool call*, JSON naming a tool and its arguments, instead of text. Your program runs the tool, such as a search or a database query, and sends the result back in the next request. `tools` was among the supported parameters of each of this chapter's three models on 2026-11-07. This chapter sends none, because its pipeline needs none, but two public standards that describe tools are built from what you used above.

**MCP**, the [Model Context Protocol](https://modelcontextprotocol.io/), is a common way for a server to offer tools to any program that uses models. Its messages are [JSON-RPC](https://www.jsonrpc.org/specification): each request is a JSON object with a `method`, `params`, and an `id` that its answer repeats. Over HTTP, a client sends each message as a `POST` and accepts the answer either as JSON or as a short event stream. [DeepWiki](https://deepwiki.com/), which writes documentation for public GitHub repositories, runs a public MCP server that needs no key. Introduce yourself with `initialize`:

In [ ]:
mcp = "https://mcp.deepwiki.com/mcp"
mcp_headers = {**HEADERS, "Accept": "application/json, text/event-stream"}
hello = {"jsonrpc": "2.0", "id": 1, "method": "initialize",
         "params": {"protocolVersion": "2025-06-18", "capabilities": {},
                    "clientInfo": {"name": "info4617", "version": "1.0"}}}
response = requests.post(mcp, json=hello, headers=mcp_headers, timeout=30)
response.raise_for_status()
print(response.headers["Content-Type"])

def sse_json(text):
    """Return the JSON on the "data: " line of a server-sent event."""
    for line in text.splitlines():
        if line.startswith("data: "):
            return json.loads(line[len("data: "):])

reply = sse_json(response.text)
print(reply["result"]["protocolVersion"], reply["result"]["serverInfo"])
# text/event-stream
# 2025-06-18 {'name': 'DeepWiki', 'version': '4.0.2'}

The server answered as a *server-sent event*: lines of text, one of which starts with `data: ` and holds the JSON. Now ask what tools it has, with `tools/list`:

In [ ]:
response = requests.post(mcp, json={"jsonrpc": "2.0", "id": 2, "method": "tools/list"}, headers=mcp_headers, timeout=30)
response.raise_for_status()
tools = sse_json(response.text)["result"]["tools"]
for tool in tools:
    print(tool["name"], tool["inputSchema"]["required"])
print(json.dumps(tools[-1]["inputSchema"]))
# ask_wiki_question ['repoName', 'question']
# read_wiki_contents ['repoName']
# read_wiki_structure ['repoName']
# {"properties": {"repoName": {"description": "GitHub repository in owner/repo format (e.g. \"facebook/react\").", "type": "string"}}, "required": ["repoName"], "type": "object", "additionalProperties": false}

Each tool's `inputSchema` is a JSON Schema, in the same language as this chapter's `SCHEMA`. The direction is reversed. `response_format` constrains what a model returns to you; an `inputSchema` constrains what a model may send to a tool. A program that connects a model to this server passes the three tools, as listed, in its request's `tools`, and when the model calls one, sends `tools/call` to the server with the model's arguments. The arguments are the model's output, so they get the same checks as any other.

The version you send decides what a request this plain can do. The specification's current version on 2026-11-07 was [2026-07-28](https://modelcontextprotocol.io/specification/versioning), which drops the `initialize` handshake: each request names its own version, and a `server/discover` request asks a server what it supports. DeepWiki answered the older handshake, from the [2025-06-18](https://modelcontextprotocol.io/specification/2025-06-18/server/tools) specification, which this section uses. A protocol this young changes quickly; read the version a server answers with before you build on it.

**Skills** are plainer still. A skill is a folder of instructions, in Markdown, that a program may give a model when a task calls for it. Its file, `SKILL.md`, starts with *front matter* between two lines of `---`: a `name`, and a `description` that says when to use it. Read the first lines of a public one, for PDFs, from Anthropic's [skills repository](https://github.com/anthropics/skills):

In [ ]:
response = requests.get("https://raw.githubusercontent.com/anthropics/skills/main/skills/pdf/SKILL.md",
                        headers=HEADERS, timeout=30)
response.raise_for_status()
skill = response.text
for line in skill.splitlines()[:5]:
    print(line[:100])
# ---
# name: pdf
# description: Use this skill whenever the user wants to do anything with PDF files. This includes rea
# license: Complete terms in LICENSE.txt
# ---

The [Agent Skills specification](https://agentskills.io/specification) describes how a program uses one: it loads every skill's name and description at the start, about a hundred tokens each, and reads the rest of a `SKILL.md` only when a task matches the description (checked 2026-11-07). The rest of this file, 314 lines in all, is a guide to Python's PDF libraries, and the first two it teaches are pypdf and pdfplumber, chapter 9's tools. A skill gives a model the same instructions a chapter gives you. It doesn't give it chapter 9's checks unless someone writes them in.

## Cost, Limits, and Reproducibility

This chapter's live run, on 2026-11-07, cost 5.06 cents of Datalab's free allowance, for the `balanced` and `lite` conversions of one report (the first request came from the cache, for nothing), and nothing at OpenRouter. The limits it ran under are the ones you will meet:

| Service | Free limit, checked 2026-11-07 | What happens at the limit | What to do |
|---|---|---|---|
| Datalab | \$10 of usage a month on a personal email (\$20 on a work email); 25 requests a minute | Pay-as-you-go rates apply | Convert once, save the Markdown, and work from the file |
| OpenRouter, `:free` models | 20 requests a minute; 50 a day on an account that has bought less than \$10 of credits, 1,000 a day on one that has bought at least \$10 | Status 429 | Pause between requests; check `GET /api/v1/key` |
| A free model's provider | Shared by everyone using it | 429 from the provider, "rate-limited upstream" | A `models` fallback list, then wait and retry |

OpenRouter's figures are from its [limits page](https://openrouter.ai/docs/api/reference/limits). A `GET` of `https://openrouter.ai/api/v1/key` with your key returns your account's `free_model_daily_requests`: how many free requests you have used today, and your limit. The provider's limit is the one a class meets: when many students send the same free model the same request in the same minute, the model's provider refuses some of them, whatever each account's own count. The fallback list is for that minute. If every model in the list is busy, the request fails with 429, and the right response is to wait a minute and try again, not to switch to a paid model.

**Reproducibility.** Everything this chapter calls can change without notice: the free models, which ones are free, the models behind Datalab's `balanced` mode, and what a model writes from one run to the next. A study that used these APIs can be described exactly and repeated approximately, and only if it records:

- the model ID that answered each request, and the date, from `result["model"]` and your own clock;
- the request bodies, including the instruction and the schema;
- the raw responses, saved as JSON before any parsing, so you can re-check them without paying again;
- the conversions, saved, with their mode, date, and cost, as `data/ch-13/manifest.csv` records this chapter's;
- the checks you ran and what they found, including the failures.

Saving outputs is cheap, and re-running is not free: it spends an allowance, and it may not give the same answer. @sec-appendix-ai-disclosure gives a template for reporting a model's use in research.

**Libraries.** This chapter uses `requests`, so that every request is visible. Both services offer libraries that hide the same requests:

- Datalab's Python library, `pip install datalab-python-sdk`, submits, waits, and returns the result in one call, `DatalabClient().convert("document.pdf")`, reading the key from `DATALAB_API_KEY` ([quickstart](https://documentation.datalab.to/docs/welcome/quickstart)).
- The `openai` library, which is in `webdata`, works with OpenRouter when you give it OpenRouter's address: `OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_key)` ([quickstart](https://openrouter.ai/docs/quickstart)). The body is the same; the library builds it from arguments.

## Recommended Exercises

This guided exercise is the chapter's take-home assignment. Work through it in the companion notebook, filling in each empty code cell, and submit the completed notebook. The steps build on one another, so do them in order; everything you need appears in this chapter or an earlier one.

You will run this chapter's pipeline on a public PDF of your choice: a government report, budget, or table that names no private individuals, ideally one with a table whose figures you can check, such as a total row or a second printing. Chapter 9's sources (@sec-pdfs) are a good place to look. Keep to Datalab's free allowance: one or two conversions are plenty.

**Step 1 — Convert your document.** Check both keys without printing them, submit your PDF to Datalab with `convert()`, and save the Markdown to a file. Split it into pages, print the page count, the cost, and one page with a table, and say in a comment which page you chose and why. Don't print a page with anyone's contact details.

In [ ]:
# Step 1: Check DATALAB_KEY and OPENROUTER_KEY, convert your PDF, save the Markdown, and print the page count, cost, and one page with a table
# Your code here

**Step 2 — Write a schema.** Choose three to five figures or facts your document prints on one page, and write a JSON Schema for them, with a type and a description for each field. Write an instruction that includes the schema, as "Seven Reports" does.

In [ ]:
# Step 2: Define your SCHEMA, and an INSTRUCTION that includes it as text
# Your code here

**Step 3 — Extract.** Send the page to a free model with `response_format`, `temperature` 0, and a `models` list, and print the model that answered and its answer. Send the same request two more times, and print each answer.

In [ ]:
# Step 3: Send the request three times, pausing between them; print the model and the answer each time
# Your code here

**Step 4 — Check.** Build your own ground truth for each field: read it from the text layer with pdfplumber (@sec-pdfs), or from the document by hand, and say which. Compare each of the model's answers with it, and check one of Datalab's tables with a sum or a second printing.

In [ ]:
# Step 4: Define your ground truth, compare the three answers with it field by field, and run one check on a table
# Your code here

**Step 5 — Retrieve with sources.** Chunk your document, embed the chunks, and ask two questions with `answer_with_sources()`: one your document answers, and one it doesn't. For each answer, check whether its value is printed where its source says.

In [ ]:
# Step 5: Chunk, embed, ask one answerable and one unanswerable question, and check each answer against the document
# Your code here

**Step 6 — Report.** In a short paragraph: what did the whole run cost, in cents and in requests, and which free models answered? Which answers were wrong, and which check caught each one? What would you have reported if you hadn't checked?

In [ ]:
# Step 6: Write your answer here as comments, or convert this cell to Markdown.

## Additional Exercises

These are open-ended extensions, with no scaffold and no fixed path.

1. **Every chart, described.** Find every chart description in the seven reports' saved Markdown (`data/ch-13/`). For each one with figures printed on the chart, check every figure in the description against the text layer with pdfplumber, and every claim of "largest" or "highest" against those figures. How many descriptions contradict their own chart?

2. **Two parsers, one page.** Convert the same page of a PDF with a parser that runs on your computer, such as Docling or MarkItDown from @sec-pdfs, and with Datalab's `lite` and `balanced` modes. Compare the three on one table with a sum check, and report what each cost in time and money.

3. **Schema or instruction.** Design a small experiment on chapter 9's seven summary pages: three versions of the request (the schema only, the schema in the instruction, and the fields described in words in the instruction without the schema), each sent three times. Report how often each version matched the ground truth, and how many reasoning tokens each spent.

4. **A different source.** Pick a public dataset that publishes both a PDF and a machine-readable file of the same figures, such as an agency's annual report and its CSV. Extract a table from the PDF with this chapter's pipeline, and measure its accuracy against the CSV, row by row.

5. **Graduate extension (INFO 5617).** Read @ziems2024can and @gilardi2023chatgpt. Then design and run a validation study for a model used as a coder in a research question of your own: choose a classification task, code 50 items with a free model through OpenRouter, and code 25 of them by hand before you look at the model's labels. Report agreement with a chance-corrected measure, such as Cohen's kappa, describe the pattern of disagreements, and run the model's 50 items a second time to report its agreement with itself. Discuss whether the papers' conclusions hold for your task, and what you would need to report so another researcher could repeat your study after your model is retired.

## Social History and Public Interest

**Public records and AI.** The city's note beside Exhibit 3 asks readers to phone for "a screen readable version": the people who need one most, those who use a screen reader because they can't see the page, can't read a picture of a table. A federal rule now says what a city's web content owes them. In April 2024 the U.S. Department of Justice issued its final rule under Title II of the Americans with Disabilities Act, which requires state and local governments' web content and mobile apps to meet the Web Content Accessibility Guidelines (WCAG) 2.1, Level AA ([89 FR 31320](https://www.federalregister.gov/documents/2024/04/24/2024-07758/nondiscrimination-on-the-basis-of-disability-accessibility-of-web-information-and-services-of-state), April 24, 2024). Governments of 50,000 people or more, Boulder among them, were to comply by April 24, 2026. On April 20, 2026, an interim final rule moved that date to April 26, 2027, and the date for smaller governments from April 26, 2027, to April 26, 2028 ([91 FR 20902](https://www.federalregister.gov/documents/2026/04/20/2026-07663/extension-of-compliance-dates-for-nondiscrimination-on-the-basis-of-disability-accessibility-of-web)). The rule exempts most PDFs posted before the compliance date, unless people still use them to apply for or reach a service ([ADA.gov](https://www.ada.gov/resources/2024-03-08-web-rule/), checked 2026-11-07).

A document-understanding API looks like a way to meet the rule at scale: convert every PDF, publish the Markdown. This chapter shows what that would publish. Datalab's reading of Exhibit 3 passed every check chapter 9 had, and could stand as the screen-readable table once a person had compared it with the image. Its description of Chart 8 contradicts the chart, and its table for the beverage chart presents 72 estimates in the form of figures. A sighted reader can look at the chart and doubt the description; a screen-reader user gets only the description. A city that publishes model-written text as its accessible version needs to say which parts a model wrote, and someone needs to check them, because the record is the city's and so is the error.

**Who checks the model.** Researchers call text that a model generates without support in its source a *hallucination*, and a survey of the problem across tasks distinguishes output that contradicts its source from output that the source can't verify [@ji2023survey]. Chart 8's description is the first kind; the beverage table is the second. Retrieval-augmented generation was proposed in part to ground a model's answers in passages it can cite [@lewis2020retrieval]. Grounding helps and doesn't settle the question. A preregistered study of commercial legal research tools built on retrieval, whose makers had advertised them as avoiding hallucinations, found that each hallucinated between 17% and 33% of the time [@magesh2025hallucination]. This chapter's \$285,000 came back with a citation to a passage that printed it; the error was in what the passage was.

Models are already used to replace some human checking. A study of annotation tasks on tweets and news articles found a model more accurate than crowd workers on most of them, at a fraction of the cost [@gilardi2023chatgpt], and a broader evaluation across computational social science tasks found models useful as coders on some tasks and unreliable on others [@ziems2024can]. Both measured the model against human labels. Chapter 9's pdfplumber figures did that job here, and they cost little because the reports have a text layer. The checks in this chapter are cheap where a document checks itself, with totals, a second printing, or a text layer, and expensive where it doesn't, which is where people most want a model to do the reading.

**Costs and labor.** A free model is not free to run. Training large models has a measurable energy and carbon cost [@strubell2019energy], and one study of running them, rather than training them, found general-purpose generative models orders of magnitude more expensive than models built for a single task, for the same task [@luccioni2024power]. This chapter's first extraction spent 2,398 tokens of reasoning to copy four numbers that pdfplumber reads in a fraction of a second on a laptop; describing the fields in the instruction cut that to a few hundred. @bender2021dangers asked researchers to weigh these environmental and financial costs, and who bears them, before reaching for the largest model. The cost of a free endpoint is also paid in data: NVIDIA's free endpoints log what you send to improve its products.

The models also rest on human work that rarely shows. Behind many automated services is *ghost work*: people paid by the task, through platforms, to label data, check outputs, and handle what the software can't [@gray2019ghost]. A study of outsourced data work for machine learning, on three platforms in Venezuela and at an outsourcing company in Argentina, found the clients' instructions, and the workers' dependence on the pay, shaping what the data said [@miceli2022data]. The labels that taught a model to tell a table from a chart were made by someone. And the concentration of data, computing, and that labor in a few companies limits what anyone outside them can inspect, even in models called open [@widder2024why]. A model whose weights you hold can't be changed or withdrawn under you, which is one reason to prefer open models for research that has to be repeated [@spirling2023why]. The **oversight** value from @sec-post-api asks the same of a model as of a city: that the people it affects can check its work.

## Common Issues to Debug

- **`KeyError` or `MISSING` for `DATALAB_KEY` or `OPENROUTER_KEY`.** Jupyter never received the variable: quit Jupyter, open a new terminal, and start it again (@sec-government, "When a Key Says `MISSING`"). Datalab's own examples read `DATALAB_API_KEY`; this book's code reads `DATALAB_KEY`.
- **Datalab answers 401 or 403.** The key is wrong or missing from the `X-API-Key` header. Print `datalab_key is None`, never the key.
- **A Datalab job that is never `complete`.** Check `success` and `error` in the result; a `file_url` that isn't public fails. The loop's limit stops your notebook from waiting forever, and the ticket's address still holds the result later.
- **A result that arrives at once, for 0 cents.** Datalab answered from its cache. Add `"skip_cache": "true"` only if you need a new conversion, and expect to pay for it.
- **`KeyError: 'markdown'`.** The result had a `result_url` instead: download the JSON from that address, without your key.
- **429 from OpenRouter.** A free model is rate-limited, for your account (20 a minute) or upstream at its provider. Use a `models` list of `:free` IDs only, pause between requests, and retry after a minute. Check your day's count with `GET /api/v1/key`.
- **`json.loads()` fails, or a field is missing.** The model that answered may not support structured outputs: check `result["model"]` against the list of models' `supported_parameters`, and put the schema in the instruction too.
- **Valid JSON, wrong figures.** Expected, sometimes. Compare with a ground truth, print the page the model read, and find where its figures came from.
- **422 from the embeddings endpoint.** An input longer than the model's limit, 4,096 tokens for this chapter's embedder: make the chunks smaller.
- **Similarity scores that make no sense.** Vectors from two different embedding models, or vectors not scaled to length 1. Embed everything with one model, and scale before `@`.
- **An MCP server answers with an error, or with no `data:` line.** Send `Accept: application/json, text/event-stream`, and parse the event stream, not `response.json()`.

## Key Takeaways

A model is an API like any other: build the request in code, ask for JSON in a shape you define, and check the answer before you use it. Datalab read Boulder's picture of a table into Markdown whose 30 rows pass chapter 9's sums, and it also wrote chart descriptions and estimates that the reports never printed; read what a parser read, and check what it wrote. A free model copied the right totals for five of seven reports and returned the wrong row and column for two, until the instruction described the fields as well as the schema. The same request at temperature 0 gave two different years. Embeddings find passages by topic, not by figure, so filter before you rank, and an answer with a source still needs its value found where the source says. Record the model, the date, the request, and the raw response, and send these services only what you would publish.

## Further Reading

- Datalab documentation: <https://documentation.datalab.to/>
- Datalab's Convert API reference: <https://documentation.datalab.to/api-reference/convert-document>
- OpenRouter documentation: <https://openrouter.ai/docs/quickstart>
- OpenRouter, Structured Outputs: <https://openrouter.ai/docs/guides/features/structured-outputs>
- OpenRouter, API rate limits: <https://openrouter.ai/docs/api/reference/limits>
- JSON Schema, getting started: <https://json-schema.org/learn/getting-started-step-by-step>
- Model Context Protocol specification: <https://modelcontextprotocol.io/specification/versioning>
- Agent Skills specification: <https://agentskills.io/specification>
- @ji2023survey — hallucination in natural language generation
- @lewis2020retrieval — the paper that named retrieval-augmented generation
- @luccioni2024power — the energy cost of running AI models